# 01 — Dataset Exploration · Phase 2

**Phase goal:** Audit all raw rasters, align them to a common grid, analyse the target distribution, and produce `baseline_dataset.csv`.

**Deliverables from this notebook:**
- Alignment audit table — which rasters need resampling
- Target distribution analysis — confirm right skew, understand BP range
- Valid-cell mask — defines the node set for Phase 3 graph construction
- `baseline_dataset.csv` saved to `data/processed/`
- `target_transformer.pkl` saved to `data/features/`

**Run `phase2_align_rasters.py` first** for a full pipeline run. This notebook is for detailed exploration and visualization.

---

In [2]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import rasterio
from pathlib import Path

# Add src/ to path
PROJECT_ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from wildfire_gnn.utils import load_yaml_config, set_seed
from wildfire_gnn.process.raster_io import (
    load_raster, load_raster_stack, audit_alignment, print_audit
)
from wildfire_gnn.process.target_engineering import (
    TargetTransformer, analyze_target_distribution
)

config = load_yaml_config(PROJECT_ROOT / 'configs' / 'gnn_config.yaml')
set_seed(config['training']['seed'])

RAW_DIR     = PROJECT_ROOT / config['paths']['raw_dir']
ALIGNED_DIR = PROJECT_ROOT / config['paths']['aligned_dir']
FEAT_DIR    = PROJECT_ROOT / config['paths']['features_dir']
PROC_DIR    = PROJECT_ROOT / 'data' / 'processed'
FIG_DIR     = PROJECT_ROOT / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Raw dir      : {RAW_DIR}')
print(f'Aligned dir  : {ALIGNED_DIR}')

Project root : d:\wildfire\spatiotemporal_wildfire_gnn
Raw dir      : d:\wildfire\spatiotemporal_wildfire_gnn\data\raw\FSim_Dataset_Greece_raw_files
Aligned dir  : d:\wildfire\spatiotemporal_wildfire_gnn\data\interim\aligned
